# Aegis Code Vulnerability Detector

**ModelHub**: MH-2026-032  
**Task**: multiclass_classification  
**Classes**: sqli / xss / cmd_injection / hardcoded_secret / path_traversal / safe  
**Input**: code snippet (string)  
**Output**: `{vuln_type: str, confidence: float}`  
**Dataset format**: JSONL `{"code": "...", "vuln_type": "safe|sqli|xss|..."}`

## GPU 需求
- Kaggle T4/P100（每週 30 hr 免費配額）
- 預計訓練時間：~3 hr（CodeBERT fine-tune）

## 產出
- `model/` — HuggingFace tokenizer + model weights
- `metrics.json` — accuracy, f1_macro, per_class_f1
- 上傳 Kaggle Dataset: `boardgamegroup/aegis-code-vuln-model-v1`


In [ ]:
# Setup
import json
import os
from pathlib import Path
from collections import Counter

DATASET_PATH = Path('/kaggle/input/aegis-code-vulnerability-dataset')
OUTPUT_PATH = Path('/kaggle/working/model')
OUTPUT_PATH.mkdir(exist_ok=True)

MODELHUB_REQ_NO = 'MH-2026-032'
MODELHUB_API_URL = os.environ.get('MODELHUB_API_URL', '')
MODELHUB_API_KEY = os.environ.get('MODELHUB_API_KEY', '')
CLASSES = ['sqli', 'xss', 'cmd_injection', 'hardcoded_secret', 'path_traversal', 'safe']


In [ ]:
# Load dataset
data_file = DATASET_PATH / 'code_vuln_dataset.jsonl'
rows = [json.loads(l) for l in data_file.read_text().splitlines() if l.strip()]
codes = [r['code'] for r in rows]
labels = [r['vuln_type'] for r in rows]
print(f'Samples: {len(codes)}')
print(f'Class distribution: {Counter(labels)}')


In [ ]:
# Train CodeBERT multiclass classifier
# TODO: 等 Kaggle dataset 上傳後補完整訓練程式碼
# 參考架構：
#   from transformers import RobertaForSequenceClassification, RobertaTokenizer, Trainer, TrainingArguments
#   tokenizer = RobertaTokenizer.from_pretrained('microsoft/codebert-base')
#   model = RobertaForSequenceClassification.from_pretrained('microsoft/codebert-base', num_labels=6)
#   ... TrainingArguments + Trainer.train() ...
print('Training placeholder — fill in after dataset upload')


In [ ]:
# Report training result to ModelHub
import urllib.request

metrics = {'accuracy': 0.0, 'f1_macro': 0.0}  # replace with real metrics

if MODELHUB_API_URL and MODELHUB_API_KEY:
    payload = json.dumps({
        'accuracy': metrics['accuracy'],
        'f1_score': metrics['f1_macro'],
        'model_path': str(OUTPUT_PATH),
        'notes': f'Kaggle T4 training, classes={CLASSES}',
    }).encode()
    req = urllib.request.Request(
        f'{MODELHUB_API_URL}/api/submissions/{MODELHUB_REQ_NO}/training-result',
        data=payload,
        headers={'x-api-key': MODELHUB_API_KEY, 'Content-Type': 'application/json'},
        method='PATCH',
    )
    with urllib.request.urlopen(req, timeout=10) as r:
        print('ModelHub updated:', r.status)
else:
    print('MODELHUB_API_URL/KEY not set, skip reporting')
